In [ ]:
import pandas as pd
import numpy as np
from typing import Union, Optional
import os

def get_time_window(
    file_path: str,
    start: Optional[Union[str, int, float]] = None,
    end: Optional[Union[str, int, float]] = None,
    window: Optional[int] = None,
) -> pd.DataFrame:
    df = pd.read_csv(file_path)
    if "created_at" in df.columns:
        df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce", utc=True)
        df = df.dropna(subset=["created_at"]).set_index("created_at").sort_index()
        if "entry_id" in df.columns:
            df["entry_id"] = pd.to_numeric(df["entry_id"], errors="coerce").astype("Int64")
        for c in [c for c in df.columns if c.startswith("field")]:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    elif "time" in df.columns:
        df.columns = [c.strip() for c in df.columns]
        df = df.dropna(subset=["time"]).set_index("time").sort_index()
        for c in [c for c in df.columns if c.startswith("s")]:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    else:
        raise ValueError("Dataset must contain either 'created_at' or 'time' column")

    df = df.ffill().bfill()
    idx = df.index
    is_dt = pd.api.types.is_datetime64_any_dtype(idx)

    if start is None and end is None:
        raise ValueError("Provide at least start or end")
    if window is not None and (not isinstance(window, (int, np.integer)) or window <= 0):
        raise ValueError("window must be a positive integer")

    if start is not None and end is not None:
        if is_dt:
            s = pd.to_datetime(start, utc=True, errors="coerce")
            e = pd.to_datetime(end,   utc=True, errors="coerce")
            if pd.isna(s) or pd.isna(e): raise ValueError("Invalid datetime for start/end")
        else:
            s = pd.to_numeric(start, errors="coerce")
            e = pd.to_numeric(end,   errors="coerce")
            if pd.isna(s) or pd.isna(e): raise ValueError("Invalid numeric start/end")
        return df.loc[s:e]

    if start is not None and window is not None:
        if is_dt:
            s = pd.to_datetime(start, utc=True, errors="coerce")
            if pd.isna(s): raise ValueError("Invalid datetime for start")
        else:
            s = pd.to_numeric(start, errors="coerce")
            if pd.isna(s): raise ValueError("Invalid numeric start")
        pos = idx.searchsorted(s)
        pos = int(pos[0]) if isinstance(pos, np.ndarray) else int(pos)
        return df.iloc[pos:pos + window]

    if end is not None and window is not None:
        if is_dt:
            e = pd.to_datetime(end, utc=True, errors="coerce")
            if pd.isna(e): raise ValueError("Invalid datetime for end")
        else:
            e = pd.to_numeric(end, errors="coerce")
            if pd.isna(e): raise ValueError("Invalid numeric end")
        pos = idx.searchsorted(e, side="right")
        pos = int(pos[0]) if isinstance(pos, np.ndarray) else int(pos)
        pos -= 1
        start_pos = max(0, pos - window + 1)
        return df.iloc[start_pos:pos + 1]

    raise ValueError("If only start or end is provided, also supply window")


def pick_examples(path, k=5):
    df = pd.read_csv(path)
    if "created_at" in df.columns:
        df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce", utc=True)
        df = df.dropna(subset=["created_at"]).sort_values("created_at")
        idx = df["created_at"].to_numpy()
    else:
        df = df.dropna(subset=["time"]).sort_values("time")
        idx = df["time"].to_numpy()
    n = len(idx)
    if n == 0: return None
    a = idx[min(2, n-1)]
    b = idx[min(2+k, n-1)]
    c = idx[min(10, n-1)]
    return a, b, c, min(k, n)

paths = ["518150.csv", "1321079.csv", "1350261.csv", "3036461.csv"]
for p in paths:
    if not os.path.exists(p):
        print(f"[Missing] {p} (upload this file to run demos)")
        continue
    ex = pick_examples(p, k=5)
    if ex is None:
        print(f"[Empty] {p}")
        continue
    a, b, c, k = ex
    print(f"\n=== {p} :: start-end slice ===")
    print(get_time_window(p, start=a, end=b).head())
    print(f"\n=== {p} :: start + window={k} ===")
    print(get_time_window(p, start=a, window=k))
    print(f"\n=== {p} :: end + window={k} ===")
    print(get_time_window(p, end=c, window=k))
